# Randomized Consistency Baseline Under Temperature Variations

This notebook starts from `4 - Evaluate LLMs under Temperature Variations.ipynb` and adds a random-permutation baseline for consistency. For each repetition file loaded as `rep_df`, each run column is independently permuted across cases. This preserves the responses available in each run while breaking case-level agreement across runs.


In [ ]:
import os
import pandas as pd
import json
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import scipy.stats as st
import matplotlib.cm as cm
import matplotlib.colorbar as colorbar
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar, Table
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import plotly.graph_objects as go


import itertools

In [ ]:
# ESAP questions with images
q_w_images = [3, 4, 6, 32, 35, 54, 68, 88]

cases = np.delete(np.arange(100), [0] + q_w_images) # valid questions
n_cases = cases.size

# Available options
options = ['A', 'B', 'C', 'D', 'E']
# even for the experiments without letter, manual annotator saved the option using letter.

path_to_results = "./../results"

model_file_name_dict = {'huatuo-o1': 'HuatuoGPT-o1',
                        'diabetica-o1': 'Diabetica-o1',
                        'diabetica-7B': 'Diabetica-7B',
                        'meditron3-8B': 'Meditron3-8B'}

In [ ]:
os.makedirs("tables", exist_ok=True)
os.makedirs("plots", exist_ok=True)


In [ ]:
model_list = ['HuatuoGPT-o1-8B', 'Meditron3-8B', 'Llama3.1-8B Instruct', 'Diabetica-o1', 'Diabetica-7B', 'Qwen2-7B Instruct']
model_file = ['huatuo-o1', 'meditron3-8B', 'llama31', 'diabetica-o1', 'diabetica-7B', 'qwen2-7b']
temperature_list = [0.3, 0.6, 1.]
path_repetitions = f"{path_to_results}/REPETITIONS"

## Consistency Helper


In [ ]:
def max_frequency_string(arr, identify_majority=True):
    """ This code returns nan if there is no majorty selected (e.g., two options with same
    frequency). Otherwise, option and how many times it was selected. """
    # with identify mayority=True, we count how many times majority was selected
    # with identify_majority False, we do not care about majority, we just want the count of the max selection
    if not arr.size:
        return None, 0
    unique_elements, counts = np.unique(arr, return_counts=True)
    max_indexes = counts == np.max(counts)
    if np.sum(max_indexes) > 1:
        if identify_majority:
            return np.nan, np.nan # returns not a number if we do not have a majority
        else:
            return np.nan, np.max(counts)
    # elif unique_elements[np.argmax(counts)] not in options:
    #     return np.nan, np.nan # if max option is none
    else:
        return unique_elements[np.argmax(counts)], np.max(counts)

## Bootstrap Helper


In [ ]:
def bootstrap_mean_ci_fast(x, B=10000, ci=95, random_state=None):
    """
    Vectorized bootstrap CI for the mean.
    """
    rng = np.random.default_rng(random_state)
    x = np.asarray(x)
    n = len(x)

    # Bootstrap resampling indices
    indices = rng.integers(0, n, size=(B, n))
    boot_samples = x[indices]
    boot_means = boot_samples.mean(axis=1)

    alpha = (100 - ci) / 2
    ci_low, ci_high = np.percentile(boot_means, [alpha, 100 - alpha])

    return x.mean(), ci_low, ci_high

## Observed Consistency


In [ ]:
# Observed consistency for each temperature, model, and case.
consistency = []

for t_ in temperature_list:
    consistency_list = []

    for m_ in model_file:
        rep_df = pd.read_excel(
            f"{path_repetitions}/T={t_}/{m_}/{m_}_promptID_001_repetitions.xlsx",
            header=None,
            index_col=0,
        )

        consistency_sel = []
        for row in rep_df.values:
            row = row[~pd.isna(row)]
            _, count = max_frequency_string(row, identify_majority=False)
            consistency_sel.append(count)
        consistency_list.append(np.array(consistency_sel))

    consistency.append(consistency_list)

consistency = np.array(consistency) / 10


In [ ]:
mean_consistency_estimate = np.zeros((len(temperature_list), len(model_list)))
consistency_lo = np.zeros_like(mean_consistency_estimate)
consistency_hi = np.zeros_like(mean_consistency_estimate)

for m in range(len(model_list)):
    for t in range(len(temperature_list)):
        mean_C, ci_lo, ci_hi = bootstrap_mean_ci_fast(
            consistency[t, m],
            B=10000,
            random_state=1000 + 100 * m + t,
        )
        print(f"Observed T={temperature_list[t]}, Model={model_list[m]}: mean={mean_C:.3f}, CI=[{ci_lo:.3f}, {ci_hi:.3f}]")
        mean_consistency_estimate[t, m] = mean_C
        consistency_lo[t, m] = ci_lo
        consistency_hi[t, m] = ci_hi
    print("-----------------")


## Randomized Complete-Inconsistency Baseline


In [ ]:
def permute_each_column(df, rng):
    """Independently permute each run column across cases."""
    random_rep_df = df.copy()
    for col in random_rep_df.columns:
        values = random_rep_df[col].to_numpy(copy=True)
        random_rep_df[col] = rng.permutation(values)
    return random_rep_df


def compute_row_consistency(rep_df):
    consistency_sel = []
    for row in rep_df.values:
        row = row[~pd.isna(row)]
        _, count = max_frequency_string(row, identify_majority=False)
        consistency_sel.append(count)
    return np.array(consistency_sel) / 10


In [ ]:
# Randomized complete-inconsistency baseline.
# Each run column is independently permuted across cases before consistency is computed.
RANDOM_SEED = 20260504
random_rng = np.random.default_rng(RANDOM_SEED)

random_consistency = []
random_permuted_repetitions = {}

for t_ in temperature_list:
    random_consistency_list = []
    random_permuted_repetitions[t_] = {}

    for m_ in model_file:
        rep_df = pd.read_excel(
            f"{path_repetitions}/T={t_}/{m_}/{m_}_promptID_001_repetitions.xlsx",
            header=None,
            index_col=0,
        )

        random_rep_df = permute_each_column(rep_df, random_rng)
        random_permuted_repetitions[t_][m_] = random_rep_df
        random_consistency_list.append(compute_row_consistency(random_rep_df))

    random_consistency.append(random_consistency_list)

random_consistency = np.array(random_consistency)


In [ ]:
random_mean_consistency_estimate = np.zeros((len(temperature_list), len(model_list)))
random_consistency_lo = np.zeros_like(random_mean_consistency_estimate)
random_consistency_hi = np.zeros_like(random_mean_consistency_estimate)

for m in range(len(model_list)):
    for t in range(len(temperature_list)):
        random_mean_C, random_ci_lo, random_ci_hi = bootstrap_mean_ci_fast(
            random_consistency[t, m],
            B=10000,
            random_state=2000 + 100 * m + t,
        )
        print(f"Random T={temperature_list[t]}, Model={model_list[m]}: mean={random_mean_C:.3f}, CI=[{random_ci_lo:.3f}, {random_ci_hi:.3f}]")
        random_mean_consistency_estimate[t, m] = random_mean_C
        random_consistency_lo[t, m] = random_ci_lo
        random_consistency_hi[t, m] = random_ci_hi
    print("-----------------")


## Summary Table


In [ ]:
random_consistency_summary = []
for t_idx, t_ in enumerate(temperature_list):
    for m_idx, model_name in enumerate(model_list):
        random_consistency_summary.append({
            "temperature": t_,
            "model": model_name,
            "observed_mean_consistency": mean_consistency_estimate[t_idx, m_idx],
            "observed_ci_low": consistency_lo[t_idx, m_idx],
            "observed_ci_high": consistency_hi[t_idx, m_idx],
            "random_mean_consistency": random_mean_consistency_estimate[t_idx, m_idx],
            "random_ci_low": random_consistency_lo[t_idx, m_idx],
            "random_ci_high": random_consistency_hi[t_idx, m_idx],
        })

random_consistency_summary = pd.DataFrame(random_consistency_summary)
random_consistency_summary


In [ ]:
os.makedirs("tables", exist_ok=True)
random_consistency_summary.to_csv("tables/random_consistency_bootstrap_summary.csv", index=False)


## Observed vs Randomized Consistency Difference

This section compares the non-permuted consistency against the randomized baseline. The `observed_minus_random` column is the direct mean difference. The bootstrap difference CI resamples case-level consistency values from both distributions.


In [ ]:
comparison_to_random = []

for t_idx, t_ in enumerate(temperature_list):
    for m_idx, model_name in enumerate(model_list):
        comparison_to_random.append({
            "temperature": t_,
            "model": model_name,
            "observed_mean": mean_consistency_estimate[t_idx, m_idx],
            "observed_ci_low": consistency_lo[t_idx, m_idx],
            "observed_ci_high": consistency_hi[t_idx, m_idx],
            "random_mean": random_mean_consistency_estimate[t_idx, m_idx],
            "random_ci_low": random_consistency_lo[t_idx, m_idx],
            "random_ci_high": random_consistency_hi[t_idx, m_idx],
            "observed_minus_random": (
                mean_consistency_estimate[t_idx, m_idx]
                - random_mean_consistency_estimate[t_idx, m_idx]
            ),
        })

comparison_to_random = pd.DataFrame(comparison_to_random)
comparison_to_random


In [ ]:
def bootstrap_mean_difference_ci(observed, random, B=10000, ci=95, random_state=None):
    """Bootstrap CI for the difference in mean consistency: observed - randomized."""
    rng = np.random.default_rng(random_state)
    observed = np.asarray(observed)
    random = np.asarray(random)

    observed_indices = rng.integers(0, len(observed), size=(B, len(observed)))
    random_indices = rng.integers(0, len(random), size=(B, len(random)))

    bootstrap_difference = (
        observed[observed_indices].mean(axis=1)
        - random[random_indices].mean(axis=1)
    )

    alpha = (100 - ci) / 2
    ci_low, ci_high = np.percentile(bootstrap_difference, [alpha, 100 - alpha])
    return bootstrap_difference.mean(), ci_low, ci_high


random_difference_rows = []

for t_idx, t_ in enumerate(temperature_list):
    for m_idx, model_name in enumerate(model_list):
        random_difference_mean, random_difference_lo, random_difference_hi = bootstrap_mean_difference_ci(
            consistency[t_idx, m_idx],
            random_consistency[t_idx, m_idx],
            B=10000,
            random_state=4000 + 100 * m_idx + t_idx,
        )

        random_difference_rows.append({
            "temperature": t_,
            "model": model_name,
            "mean_difference_observed_minus_random": random_difference_mean,
            "difference_ci_low": random_difference_lo,
            "difference_ci_high": random_difference_hi,
            "ci_excludes_zero": (random_difference_lo > 0) or (random_difference_hi < 0),
        })

random_difference_df = pd.DataFrame(random_difference_rows)
random_difference_df


In [ ]:
comparison_to_random.to_csv("tables/observed_vs_random_consistency_summary.csv", index=False)
random_difference_df.to_csv("tables/observed_minus_random_consistency_bootstrap_ci.csv", index=False)


## Random Baseline vs Observed Temperatures


In [ ]:
fig, ax = plt.subplots(figsize=(18, 3), ncols=len(model_list), sharey=True)

colors_list = [
    "dimgray",               # randomized baseline
    (0.3060, 0.4285, 0.7891), # T = 0.3
    "lightsteelblue",        # T = 0.6
    "gold",                  # T = 1.0
]
plot_labels = ["Random", "0.3", "0.6", "1.0"]
x_positions = np.arange(4)

for i, model_name in enumerate(model_list):
    for spine in ax[i].spines.values():
        spine.set_linewidth(2)
    ax[i].tick_params(width=2)
    ax[i].grid(axis="y", alpha=0.4)

    # Random baseline averaged over the three source temperature files for this model.
    random_model_mean, random_model_lo, random_model_hi = bootstrap_mean_ci_fast(
        random_consistency[:, i, :].ravel(),
        B=10000,
        random_state=3000 + i,
    )
    ax[i].errorbar(
        0,
        random_model_mean,
        yerr=[[random_model_mean - random_model_lo], [random_model_hi - random_model_mean]],
        fmt="o",
        color=colors_list[0],
        ecolor=colors_list[0],
        elinewidth=3,
        capsize=6,
        capthick=3,
        markersize=8,
        zorder=5,
    )

    for t_idx, t_ in enumerate(temperature_list):
        mean = mean_consistency_estimate[t_idx, i]
        lo = consistency_lo[t_idx, i]
        hi = consistency_hi[t_idx, i]
        ax[i].errorbar(
            t_idx + 1,
            mean,
            yerr=[[mean - lo], [hi - mean]],
            fmt="o",
            color=colors_list[t_idx + 1],
            ecolor=colors_list[t_idx + 1],
            elinewidth=3,
            capsize=6,
            capthick=3,
            markersize=8,
            zorder=5,
        )

    ax[i].set_title(model_name, fontsize=15, weight="bold")
    ax[i].set_xticks(x_positions, plot_labels, fontsize=12)
    ax[i].set_ylim(0.0, 1.0)
    ax[i].set_xlim([-0.4, 3.4])

ax[0].tick_params(axis="y", labelsize=14)
ax[0].set_ylabel("Mean Consistency", fontsize=16)
fig.supxlabel("Baseline / Temperature", fontsize=16, y=0.02)

plt.tight_layout()
plt.savefig("plots/random_vs_observed_consistency_bootstrap_ci.pdf")
plt.show()
